# ConvNeXt M3 — resume QAT epoch 2 trên Kaggle T4 x2
Notebook tải FP32 và `qat_last.pt` epoch 1, sau đó train đúng một epoch để tạo checkpoint epoch 2. Calibration không chạy lại khi resume.

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Bật Accelerator: GPU T4 x2 và Internet trong Kaggle Settings'
assert torch.cuda.device_count() >= 2, f'Cần T4 x2, hiện chỉ thấy {torch.cuda.device_count()} GPU'
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), 'VRAM GB:', torch.cuda.get_device_properties(i).total_memory / 2**30)

## 1. Clone project và cài dependencies

In [ ]:
REPO = Path('/kaggle/working/EchteAI')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/NguyenDucThang-tb/EchteAI.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[coco]', 'psutil', 'kagglehub'], check=True)
import kagglehub
print('Repository:', Path.cwd())

## 2. Cấu hình nguồn checkpoint

In [ ]:
KAGGLE_USERNAME = 'nguyenducthangtb'
CHECKPOINT_DATASET = f'{KAGGLE_USERNAME}/echteai-seadronessee-m3-checkpoints'
RESULT_DATASET = f'{KAGGLE_USERNAME}/echteai-seadronessee-m3-qat-epoch2'
SEADRONESSEE_DATASET = 'ubiratanfilho/sds-dataset'
VARIANT = 'M3'
print('FP32 + QAT source:', CHECKPOINT_DATASET)
print('Output:', RESULT_DATASET)

## 3. Tải dataset và kiểm tra checkpoint epoch 1

In [ ]:
def find_dataset_root(start):
    start = Path(start)
    for candidate in [start, *[p.parent.parent for p in start.rglob('instances_train.json')]]:
        if (candidate/'annotations/instances_train.json').exists() and (candidate/'images/train').is_dir():
            return candidate
    raise FileNotFoundError(f'Không tìm thấy SeaDronesSee trong {start}')

DATA_ROOT = find_dataset_root(kagglehub.dataset_download(SEADRONESSEE_DATASET))
CHECKPOINT_DOWNLOAD = Path(kagglehub.dataset_download(CHECKPOINT_DATASET, force_download=True))
fp32_matches = list(CHECKPOINT_DOWNLOAD.rglob('fp32_best.pt'))
qat_matches = list(CHECKPOINT_DOWNLOAD.rglob('qat_last.pt'))
assert fp32_matches, f'Không tìm thấy fp32_best.pt trong {CHECKPOINT_DOWNLOAD}'
assert qat_matches, f'Không tìm thấy qat_last.pt trong {CHECKPOINT_DOWNLOAD}. Hãy upload qat_last.pt vào version mới nhất.'
FP32_CHECKPOINT = fp32_matches[0]
QAT_RESUME_CHECKPOINT = qat_matches[0]
resume_payload = torch.load(QAT_RESUME_CHECKPOINT, map_location='cpu', weights_only=False)
assert int(resume_payload.get('epoch', -1)) == 1, f'Checkpoint phải là epoch 1, thực tế={resume_payload.get("epoch")}'
print('Dataset root:', DATA_ROOT)
print('FP32 checkpoint:', FP32_CHECKPOINT)
print('QAT resume checkpoint:', QAT_RESUME_CHECKPOINT)
print('Resume epoch:', resume_payload.get('epoch'))
print('Resume metrics:', resume_payload.get('metrics', {}))

## 4. Tạo runtime config epoch 2/11

In [ ]:
import yaml
config = yaml.safe_load(Path('configs/seadronessee_colab.yaml').read_text())
OUTPUT = Path('/kaggle/working/qat_epoch2_m3')
OUTPUT.mkdir(parents=True, exist_ok=True)
config['dataset'].update({
    'train_images': str(DATA_ROOT/'images/train'), 'train_annotations': str(DATA_ROOT/'annotations/instances_train.json'),
    'val_images': str(DATA_ROOT/'images/val'), 'val_annotations': str(DATA_ROOT/'annotations/instances_val.json'),
    'test_images': str(DATA_ROOT/'images/val'), 'test_annotations': str(DATA_ROOT/'annotations/instances_val.json'),
})
DDP_GPUS = 2
QAT_BATCH_PER_GPU = 1  # global batch = 2; chỉ tăng lên 2 nếu chắc chắn không OOM
config['training']['qat_epochs'] = 11
config['training']['qat_batch_size'] = QAT_BATCH_PER_GPU
config['training']['print_frequency'] = 5
config['quantization']['weight_only_warmup_epochs'] = 0
config['quantization']['observer_freeze_epochs'] = 1
config['quantization']['phase_learning_rates']['full'] = 1e-5
config['output'] = {
    'directory': str(OUTPUT), 'fp32_best': str(FP32_CHECKPOINT), 'fp32_last': str(FP32_CHECKPOINT),
    'qat_best': str(OUTPUT/'qat_best.pt'), 'qat_last': str(OUTPUT/'qat_last.pt'),
    'int8_model': str(OUTPUT/'selective_int8.pt'), 'evaluation_json': str(OUTPUT/'evaluation.json'),
    'benchmark_json': str(OUTPUT/'benchmark.json'), 'epoch_benchmarks': str(OUTPUT/'epoch_benchmarks.json'),
}
RUNTIME_CONFIG = Path('/kaggle/working/qat_epoch2_runtime.yaml')
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print('Runtime config:', RUNTIME_CONFIG)
print('Batch/GPU:', QAT_BATCH_PER_GPU, 'global batch:', QAT_BATCH_PER_GPU * DDP_GPUS)

## 5. Resume epoch 1 và train đúng epoch 2

In [ ]:
command = [
    sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node', str(DDP_GPUS),
    'scripts/train_qat_ddp.py', '--config', str(RUNTIME_CONFIG),
    '--fp32-checkpoint', str(FP32_CHECKPOINT), '--resume', str(QAT_RESUME_CHECKPOINT),
    '--variant', VARIANT, '--epochs-this-run', '1', '--no-find-unused-parameters',
]
print('Command:', ' '.join(command), flush=True)
log_path = OUTPUT/'qat_train_ddp.log'
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
with log_path.open('a', encoding='utf-8') as log_file:
    log_file.write('\n===== QAT DDP EPOCH 2/11 START =====\n'); log_file.flush()
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in process.stdout:
        print(line, end='', flush=True); log_file.write(line); log_file.flush()
    code = process.wait()
if code:
    raise subprocess.CalledProcessError(code, command)

## 6. Xác nhận epoch 2 và upload checkpoint

In [ ]:
QAT_EPOCH2 = OUTPUT/'qat_last.pt'
assert QAT_EPOCH2.exists(), 'Không tìm thấy qat_last.pt sau khi train'
epoch2_payload = torch.load(QAT_EPOCH2, map_location='cpu', weights_only=False)
assert int(epoch2_payload.get('epoch', -1)) == 2, f'Kỳ vọng epoch=2, thực tế={epoch2_payload.get("epoch")}'
print('Verified checkpoint:', QAT_EPOCH2)
print('Epoch:', epoch2_payload.get('epoch'))
print('Metrics:', epoch2_payload.get('metrics', {}))
for path in sorted(OUTPUT.glob('*')):
    print(f'{path.name:30s} {path.stat().st_size/2**20:9.2f} MB')
kagglehub.dataset_upload(RESULT_DATASET, str(OUTPUT), version_notes='M3 QAT epoch 2/11 resumed from qat_last epoch 1')
print('Checkpoint epoch 2 uploaded:', RESULT_DATASET)